In [ ]:
import os

INPUT_BASE = '/kaggle/input/datasets/msebastiaanh/5gcn-2905/HAABSA7GCN'

assert os.path.isdir(f'{INPUT_BASE}/src'), f"src/ not found under {INPUT_BASE}"
assert os.path.isdir(f'{INPUT_BASE}/data'), f"data/ not found under {INPUT_BASE}"
print('src/:', sorted(os.listdir(f'{INPUT_BASE}/src')))
print('data/:', sorted(os.listdir(f'{INPUT_BASE}/data')))

In [ ]:
import shutil, os

WORK = '/kaggle/working'
os.makedirs(f'{WORK}/data', exist_ok=True)

# Copy all data files (including .const, .dep, ontology, rem*.csv)
for f in os.listdir(f'{INPUT_BASE}/data'):
    shutil.copy(f'{INPUT_BASE}/data/{f}', f'{WORK}/data/{f}')

# cooc_matrix_final2.csv sits at the top of the bundle (alongside src/ and data/)
shutil.copy(f'{INPUT_BASE}/cooc_matrix_final2.csv', f'{WORK}/cooc_matrix_final2.csv')

# test_ontology_keys.csv is under src/
shutil.copy(f'{INPUT_BASE}/src/test_ontology_keys.csv', f'{WORK}/test_ontology_keys.csv')

os.chdir(WORK)
print('cwd:', os.getcwd())
print('working/ contents:', sorted(os.listdir(WORK)))

# Verify the .const files made it
for year in ['2015', '2016']:
    for split in ['train', 'test']:
        p = f'data/{split}{year}restaurant.txt.const'
        assert os.path.exists(p), f'MISSING: {p}'
        with open(p) as f:
            n = sum(1 for _ in f)
        print(f'  {p}: {n} lines')

# Also confirm cooc + ontology landed
assert os.path.exists('cooc_matrix_final2.csv'), 'cooc matrix missing'
assert os.path.exists('test_ontology_keys.csv'), 'ontology keys missing'
print('cooc + ontology OK')

In [ ]:
!pip install pytorch_transformers==1.2.0 --quiet
!pip install hyperopt --quiet

In [ ]:
import torch, sys
print('python:', sys.version.split()[0])
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('CUDA device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')

import pytorch_transformers
print('pytorch_transformers:', pytorch_transformers.__version__)

In [ ]:
import nbformat

NB_PATH = f'{INPUT_BASE}/src/7GCN.ipynb'
nb = nbformat.read(NB_PATH, as_version=4)

DEFINITION_CELLS = list(range(0, 25))

for i, cell in enumerate(nb.cells):
    if i not in DEFINITION_CELLS:
        continue
    if cell.cell_type != 'code':
        continue
    src = ''.join(cell.source)
    if not src.strip():
        continue
    try:
        exec(src, globals())
        print(f"✓ cell {i:2d} OK ({len(src)} chars)")
    except Exception as e:
        print(f"✗ cell {i:2d} FAILED: {type(e).__name__}: {e}")
        raise

In [ ]:
import pandas as pd

print('Loading co-occurrence matrix...')
cooc = pd.read_csv('cooc_matrix_final2.csv', index_col=0)
print(f'  cooc shape: {cooc.shape}')

print('Loading ontology words...')
onto_words_df = pd.read_csv('test_ontology_keys.csv', sep=';')
onto_words = [item for sublist in onto_words_df.values.tolist() for item in sublist]
onto_words = list(dict.fromkeys(onto_words))
print(f'  ontology words count: {len(onto_words)}')

In [ ]:
import time, pickle, os
from math import log
from functools import partial
from hyperopt import hp, tpe, fmin, Trials, STATUS_OK
from hyperopt.fmin import generate_trials_to_calculate

# ---- search space (locked) -------------------------------------------------
space = [
    hp.loguniform('learning_rate', log(8e-6), log(3e-5)),  # brackets inherited 1.11e-5
    hp.quniform('dropout', 0.15, 0.35, 0.05),              # around inherited 0.229
    hp.loguniform('l2reg', log(5e-4), log(2e-2)),          # inherited 0.027 ABOVE top: search downward
]

CONCAT_DROPOUT_FIXED = 0.2285714285714286
SEED, YEAR, EPOCHS   = 7, '2015', 6
TMP_MODEL_PATH       = '/kaggle/working/tpe_tmp'   # throwaway, overwritten each trial
PKL                  = '/kaggle/working/tpe_4gcn_2015_seed7.pkl'
WALL_LIMIT           = 11 * 3600                   # stop before Kaggle's 12h cap

# inherited config as the first forced evaluation (nearest in-range values)
SEED_POINTS = [{'learning_rate': 1.1122448979591838e-05,
                'dropout': 0.20,     # nearest 0.05-grid value to 0.229
                'l2reg': 0.02}]      # nearest in-range value to inherited 0.027

START = time.time()
trial_times = []

# ---- objective -------------------------------------------------------------
def objective(params):
    lr, dropout, l2reg = params
    t0 = time.time()
    global opt
    opt = get_args(
        model_type='tri_gcn', constgcn=False,
        tgcn=True, semgcn=True, lexgcn=True, knogcn=True,
        year=YEAR, seed=SEED,
        learning_rate=float(lr), dropout=float(dropout),
        concat_dropout=CONCAT_DROPOUT_FIXED, l2reg=float(l2reg),
        num_epoch=EPOCHS, batch_size=4,
        save_models='none', fusion_type='concat', use_ensemble=False,
        cooc=cooc, onto_words=onto_words,
        do_train=True, do_eval=True,
        path=TMP_MODEL_PATH,
    )
    opt.device = torch.device('cuda')
    max_val_acc, test_acc, test_f1 = main(opt)
    dt = time.time() - t0
    trial_times.append(dt)
    print(f'[trial] lr={lr:.3e} drop={dropout:.2f} l2={l2reg:.3e} '
          f'-> val={max_val_acc:.4f} test={test_acc:.4f} ({dt/60:.1f} min)')
    return {'loss': -max_val_acc, 'status': STATUS_OK, 'space': params,
            'val_acc': float(max_val_acc), 'test_acc': float(test_acc),
            'test_f1': float(test_f1)}

# ---- resume or seed --------------------------------------------------------
def n_done(tr):
    return len([t for t in tr.trials if t['state'] == 2])

try:
    trials = pickle.load(open(PKL, 'rb'))
    print(f'Resumed {n_done(trials)} completed trials from {PKL}')
except (FileNotFoundError, EOFError):
    trials = generate_trials_to_calculate(SEED_POINTS)   # seeds the first trial
    print('Fresh Trials; first evaluation seeded with the inherited config.')

algo = partial(tpe.suggest, n_startup_jobs=5)

# ---- search loop with wall-clock guard ------------------------------------
while True:
    elapsed = time.time() - START
    est = max(trial_times) if trial_times else 45 * 60      # 45 min prior until measured
    if elapsed + est > WALL_LIMIT:
        print(f'STOP: {elapsed/3600:.2f}h elapsed; next trial (~{est/60:.0f} min) '
              f'would exceed the {WALL_LIMIT/3600:.0f}h guard.')
        break
    n = n_done(trials) + 1
    fmin(objective, space=space, algo=algo, trials=trials,
         max_evals=n, show_progressbar=False)
    pickle.dump(trials, open(PKL, 'wb'))                     # persist after EVERY trial
    
    import json
    done_trials = [t for t in trials.trials if t['state'] == 2]
    best_t = min(done_trials, key=lambda t: t['result']['loss'])
    bl, bd, b2 = best_t['result']['space']
    with open('/kaggle/working/tpe_best_so_far.json', 'w') as f:
        json.dump({'val_acc': best_t['result']['val_acc'],
                   'test_acc': best_t['result'].get('test_acc'),
                   'learning_rate': float(bl), 'dropout': float(bd),
                   'l2reg': float(b2),
                   'concat_dropout': 0.2285714285714286,
                   'n_trials_done': len(done_trials)}, f, indent=2)
        
    best_val = -min(t['result']['loss'] for t in trials.trials if t['state'] == 2)
    print(f'== {n_done(trials)} trials done; best val_acc so far = {best_val:.4f} ==')

print(f'\nSearch ended: {n_done(trials)} trials completed. '
      f'Run the summary cell to read the winner.')

In [ ]:
import pickle, json

# TPE summary, results sorted by validation accuracy.

PKL = '/kaggle/working/tpe_4gcn_2015_seed7.pkl'
trials = pickle.load(open(PKL, 'rb'))

rows = []
for t in trials.trials:
    if t['state'] != 2:
        continue
    r = t['result']
    if r.get('status') != STATUS_OK:
        continue
    lr, drop, l2 = r['space']
    rows.append({
        'val_acc':  r['val_acc'],
        'test_acc': r.get('test_acc'),
        'test_f1':  r.get('test_f1'),
        'lr':       float(lr),
        'dropout':  float(drop),
        'l2reg':    float(l2),
    })

rows.sort(key=lambda x: x['val_acc'], reverse=True)

print(f'{len(rows)} completed trials (sorted by val_acc):\n')
print(f"{'rank':>4} {'val_acc':>8} {'test_acc':>9} {'lr':>11} {'dropout':>8} {'l2reg':>10}")
for i, r in enumerate(rows, 1):
    ta = r['test_acc'] if r['test_acc'] is not None else float('nan')
    print(f"{i:>4} {r['val_acc']:>8.4f} {ta:>9.4f} "
          f"{r['lr']:>11.3e} {r['dropout']:>8.2f} {r['l2reg']:>10.3e}")

if rows:
    best = rows[0]
    print('\n' + '=' * 60)
    print('BEST CONFIG (by validation accuracy):')
    print(f"  learning_rate  = {best['lr']:.6e}")
    print(f"  dropout        = {best['dropout']:.2f}")
    print(f"  l2reg          = {best['l2reg']:.6e}")
    print(f"  concat_dropout = {0.2285714285714286:.6f}   (held fixed)")
    print(f"  -> val_acc {best['val_acc']:.4f}  (6-epoch search, seed 7, 2015)")
    print('=' * 60)
    print('\nNEXT: validate this config at full 15 epochs, seed 7, on BOTH')
    print('years (2015 + 2016) in a second session.')
    with open('/kaggle/working/tpe_best_config.json', 'w') as f:
        json.dump(best, f, indent=2)
    print('\nWrote /kaggle/working/tpe_best_config.json')